# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

 **My Rule:**<br> If CTR is low, engagement is low, impressions are less, and position is out of 10 than refresh page- here "refresh page" output acts as decision support where the human has to interfere and look at the page to see what specific problem is there and fix it.

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import pandas as pd
import os, getpass
import duckdb

In [7]:
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [9]:
con.sql(f"SUMMARIZE (SELECT scroll_events FROM {TABLES['fact_daily']})").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬─────────────┬─────────┬─────────┬───────────────┬─────────────────────┬────────────────────┬─────────┬─────────┬─────────┬──────────┬─────────────────┐
│  column_name  │ column_type │   min   │   max   │ approx_unique │         avg         │        std         │   q25   │   q50   │   q75   │  count   │ null_percentage │
│    varchar    │   varchar   │ varchar │ varchar │     int64     │       varchar       │      varchar       │ varchar │ varchar │ varchar │  int64   │  decimal(9,2)   │
├───────────────┼─────────────┼─────────┼─────────┼───────────────┼─────────────────────┼────────────────────┼─────────┼─────────┼─────────┼──────────┼─────────────────┤
│ scroll_events │ BIGINT      │ 0       │ 3053    │           157 │ 0.02231275775234669 │ 1.8311748287481193 │ 0       │ 0       │ 0       │ 78835655 │           37.59 │
└───────────────┴─────────────┴─────────┴─────────┴───────────────┴─────────────────────┴────────────────────┴─────────┴─────────┴─────────┴──────────

In [12]:
reference_date = con.sql(f"""
    SELECT MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
""").fetchone()[0]
print(f"Reference date set to: {reference_date}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Reference date set to: 2026-06-30


In [23]:
query = f"""
    SELECT
        f.*,
        d.content_updated_date,
        f.clicks / NULLIF(f.gsc_impressions, 0) AS ctr,
        f.ga4_engaged_sessions / NULLIF(f.ga4_sessions, 0) AS engagement_rate,
        f.scroll_events / NULLIF(f.ga4_pageviews, 0) AS scroll_rate,
        DATE_DIFF('day', CAST(d.content_updated_date AS DATE), DATE '{reference_date}') AS days_since_update
    FROM {TABLES['fact_daily_sample']} f
    LEFT JOIN {TABLES['dim_content']} d
        ON f.content_hash_id = d.content_hash_id
"""

In [24]:
print(query)


    SELECT
        f.*,
        d.content_updated_date,
        f.clicks / NULLIF(f.gsc_impressions, 0) AS ctr,
        f.ga4_engaged_sessions / NULLIF(f.ga4_sessions, 0) AS engagement_rate,
        f.scroll_events / NULLIF(f.ga4_pageviews, 0) AS scroll_rate,
        DATE_DIFF('day', CAST(d.content_updated_date AS DATE), DATE '2026-06-30') AS days_since_update
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet') f
    LEFT JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
        ON f.content_hash_id = d.content_hash_id



In [25]:
import os
os.makedirs('work/outputs', exist_ok=True)

# Execute query and save to parquet
con.sql(f"COPY ({query}) TO 'updated_features.parquet' (FORMAT PARQUET)")
# Load into a dataframe for the next steps
data = con.sql("SELECT * FROM 'updated_features.parquet'").df()
print(f"Success! Data loaded. Shape: {data.shape}")
display(data.head())


BinderException: Binder Error: Table "f" does not have a column named "clicks"

Candidate bindings: : "client_hash_id"

LINE 5:         f.clicks / NULLIF(f.gsc_impressions, 0) AS ctr,
                ^

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
reference_date = data["report_date"].max()
data["ctr"] = data["clicks"] / data["gsc_impressions"]
data["engagement_rate"] = data["ga4_engaged_sessions"] / data["ga4_sessions"]
data["scroll_rate"] = data["scroll_events"] / data["ga4_pageviews"]
data["days_since_update"] = (reference_date - data["content_updated_date"]).dt.days


if data["gsc_avg_position"]<20 and data["ctr"]<0.4 and data["engagement_rate"]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.